In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
print("Available variables:")

print([name for name in globals() if not name.startswith('_')])

In [ ]:
print("BOOKS COLUMNS:")
print(df_books.columns.tolist())

print("\nRATINGS COLUMNS:")
print(df_ratings.columns.tolist())

print("\nBOOKS PREVIEW:")
display(df_books.head(3))

print("\nRATINGS PREVIEW:")
display(df_ratings.head(3))

In [ ]:

# =====================================================
# BOOK RECOMMENDATION ENGINE USING K-NEAREST NEIGHBORS
# =====================================================

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# 1. Filter users with at least 200 ratings

user_counts = df_ratings['user'].value_counts()

filtered_ratings = df_ratings[
    df_ratings['user'].isin(
        user_counts[user_counts >= 200].index
    )
].copy()


# 2. Merge ratings with book information

df = pd.merge(
    filtered_ratings,
    df_books,
    on='isbn'
)


# 3. Filter books with at least 100 ratings

book_counts = df['title'].value_counts()

df = df[
    df['title'].isin(
        book_counts[book_counts >= 100].index
    )
].copy()


# 4. Create the user-book rating matrix

book_user_matrix = df.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)


# 5. Train the KNN model

model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(book_user_matrix.values)


# 6. Define the recommendation function

def get_recommends(book):

    if book not in book_user_matrix.index:
        raise ValueError(f"Book not found: {book}")

    # Find the six nearest neighbors
    distances, indices = model.kneighbors(
        book_user_matrix.loc[book].values.reshape(1, -1),
        n_neighbors=6
    )

    # Pair book indices with distances
    results = list(zip(
        indices[0].tolist(),
        distances[0].tolist()
    ))

    # Remove the original book
    results = results[1:]

    # Sort by descending distance as required by FCC
    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Build recommendations
    recommended_books = []

    for index, distance in results:

        recommended_books.append([
            book_user_matrix.index[index],
            distance
        ])

    return [book, recommended_books]


# 7. Display dataset and model information

print("Number of books:", book_user_matrix.shape[0])
print("Number of users:", book_user_matrix.shape[1])
print("Model trained successfully!")


In [ ]:
print("Rating matrix dimensions:", book_user_matrix.shape)

print("Model features:", model.n_features_in_)

print(
    "Features match:",
    book_user_matrix.shape[1] == model.n_features_in_
)

In [ ]:
books = get_recommends(
    "Where the Heart Is (Oprah's Book Club (Paperback))"
)

print(books)

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

test = get_recommends("The Queen of the Damned (Ann Rice)")
print(test)

In [ ]:
# Check whether the Queen title exists in the books dataset
queen_books = df_books[
    df_books['title'].str.contains(
        'Queen of the Damned',
        case=False,
        na=False
    )
]

print("Matching titles in df_books:")
print(queen_books[['isbn', 'title']])

# Check whether it exists in the rating matrix
print("\nMatching titles in book_user_matrix:")
print([
    title for title in book_user_matrix.index
    if 'Queen of the Damned' in title
])

# Check which matrix the current model uses
print("\nMatrix dimensions:", book_user_matrix.shape)
print("Model features:", model.n_features_in_)

# Check the active function
import inspect
print("\nCurrent get_recommends function:")
print(inspect.getsource(get_recommends))

In [ ]:
# Get all ISBNs associated with Queen of the Damned
queen_isbns = queen_books['isbn'].unique()

# Check their rating counts in the original ratings dataset
queen_rating_counts = (
    df_ratings[df_ratings['isbn'].isin(queen_isbns)]
    .groupby('isbn')
    .agg(
        rating_count=('rating', 'count'),
        unique_users=('user', 'nunique')
    )
    .reset_index()
)

print("Queen of the Damned rating counts:")
print(queen_rating_counts.to_string(index=False))

# Check the most frequently rated books
print("\nTop 15 ISBNs by rating count:")
print(
    df_ratings['isbn']
    .value_counts()
    .head(15)
)

In [ ]:
print("Original ratings:", df_ratings.shape)
print("Current recommendation matrix:", book_user_matrix.shape)

print("\nCurrent matrix titles:")
print(book_user_matrix.index.tolist())

In [ ]:
# Identify users with at least 200 ratings
active_users = df_ratings['user'].value_counts()
active_users = active_users[active_users >= 200].index

# Keep ratings from active users
filtered_ratings = df_ratings[
    df_ratings['user'].isin(active_users)
].copy()

# Check the Queen ISBN after user filtering
queen_after_filter = (
    filtered_ratings[
        filtered_ratings['isbn'].isin(queen_isbns)
    ]
    .groupby('isbn')
    .agg(
        rating_count=('rating', 'count'),
        unique_users=('user', 'nunique')
    )
    .reset_index()
)

print("Ratings remaining after active-user filtering:")
print(queen_after_filter.to_string(index=False))

# Check whether the main Queen edition survives the 100-rating threshold
main_queen_isbn = '0345351525'

print(
    "\nMain Queen edition ratings after filtering:",
    len(filtered_ratings[filtered_ratings['isbn'] == main_queen_isbn])
)

# Check the exact book title and author
print("\nMain Queen edition details:")
print(
    df_books[df_books['isbn'] == main_queen_isbn][
        ['isbn', 'title', 'author']
    ].to_string(index=False)
)

In [ ]:

def get_recommends(book):

    # Check that the book exists
    if book not in book_ratings.index:
        raise ValueError(f"Book not found: {book}")

    # Get the book index
    book_index = book_ratings.index.get_loc(book)

    # Find the nearest neighbors
    distances, indices = model.kneighbors(
        book_matrix[book_index],
        n_neighbors=6
    )

    # Create the recommendation list
    recommended_books = []

    # Skip the first result (the original book)
    for i in range(1, 6):

        recommended_title = book_ratings.index[indices[0][i]]
        recommended_distance = distances[0][i]

        recommended_books.append([
            recommended_title,
            recommended_distance
        ])

    # Reverse the list to match FreeCodeCamp's expected order
    recommended_books.reverse()

    return [book, recommended_books]


In [ ]:

def get_recommends(book):

    distances, indices = model.kneighbors(
        book_user_matrix.loc[book].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []

    for i in range(1, 6):
        recommended_books.append([
            book_user_matrix.index[indices[0][i]],
            distances[0][i]
        ])

    recommended_books.reverse()

    return [book, recommended_books]


In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()